# CheXpert preprocessing

This notebook prepares CheXpert as a separate metadata-only federated client. For the configurable target, `1` is positive and `0` is negative; `-1` uncertain and blank/not mentioned are excluded. They are never silently converted to zero.

In [ ]:
from pathlib import Path
import json, sys, platform
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir() and Path('/content').exists(): PROJECT_ROOT = Path('/content') / 'Project Repo'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from config import TARGET_DISEASE, RANDOM_SEED, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES
from dataset_utils import chexpert_clean_binary_labels, standardize_chexpert
from preprocessing_utils import (assert_no_leakage, assert_no_study_leakage, build_summary, require_directory, require_file, sample_clean_cases, save_splits, save_summary, set_reproducibility, split_by_patient)
set_reproducibility(RANDOM_SEED)
print('Python:', platform.python_version(), 'pandas:', pd.__version__)
CHEXPERT_ROOT = PROJECT_ROOT / 'data' / 'CheXpert'
TRAIN_CSV = CHEXPERT_ROOT / 'train.csv'
IMAGE_ROOT = CHEXPERT_ROOT
OUTPUT_DIR = PROJECT_ROOT / 'processed' / 'client_02_CheXpert'
print({'CHEXPERT_ROOT': CHEXPERT_ROOT, 'TRAIN_CSV': TRAIN_CSV, 'IMAGE_ROOT': IMAGE_ROOT, 'OUTPUT_DIR': OUTPUT_DIR})

## Load and inspect
CheXpert contains one label column per pathology. We select the configured target column exactly; release-specific column names should be verified before execution.

In [ ]:
csv_path = require_file(TRAIN_CSV, 'CheXpert train metadata CSV')
image_root = require_directory(IMAGE_ROOT, 'CheXpert dataset root')
chexpert = pd.read_csv(csv_path)
target_column = TARGET_DISEASE
if target_column not in chexpert.columns:
    raise KeyError(f'CheXpert target column {target_column!r} is absent. Available columns: {chexpert.columns.tolist()}')
print('studies/images:', len(chexpert))
print('columns:', chexpert.columns.tolist())
display(chexpert.isna().sum().sort_values(ascending=False).head(25).to_frame('missing'))
display(chexpert[target_column].value_counts(dropna=False).sort_index().to_frame('raw_target_distribution'))
clean, counts = chexpert_clean_binary_labels(chexpert, target_column)
print('Counts before filtering:', counts)

In [ ]:
standardized = standardize_chexpert(clean, image_root)
standardized = sample_clean_cases(standardized, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES, RANDOM_SEED)
print('Clean rows after optional sampling:', len(standardized))
print('unique patients:', standardized['patient_id'].nunique())
display(standardized.head())
splits = split_by_patient(standardized, seed=RANDOM_SEED)
assert_no_leakage(splits)
assert_no_study_leakage(splits)
print('patient and study leakage checks: passed')
save_splits(splits, OUTPUT_DIR)
summary = build_summary(splits, 'CheXpert', {'target_disease': TARGET_DISEASE, 'random_seed': RANDOM_SEED, 'max_positive_samples': MAX_POSITIVE_SAMPLES, 'max_negative_samples': MAX_NEGATIVE_SAMPLES, 'filter_counts_before_filtering': counts})
save_summary(summary, OUTPUT_DIR)
print(json.dumps(summary, indent=2))
display(pd.concat(splits).groupby('target').size().rename('count').to_frame())